# Async-await, Promise and Events

In [1]:
// Import the EventEmitter class from the 'events' module
const EventEmitter = require('events');

## How **Node.js** Works Asynchronously: The **Event Loop**
- Node.js has an **event-driven**, **non-blocking** I/O model, which makes it highly efficient for handling many concurrent operations.
- At the core of this behavior is the Node.js **Event Loop**, which manages the execution of asynchronous code, events, timers, and I/O operations.
- It handles concurrent I/O operations efficiently by offloading them to the system (or worker threads) while allowing other code to continue executing.

> The Node.js has an **Event Loop** that consists of **various phases**, and in **between these phases**, the **Microtask Queue** is processed.

- **Phases of the Event Loop**: The Event Loop is divided into different phases, and each phase has a specific role in handling various types of callbacks. The key phases are:
  1. **Timers Phase**: Handles the execution of **callbacks** scheduled by `setTimeout()` and `setInterval()`.
  2. **I/O Callbacks Phase**: This phase is dedicated to executing asynchronous **I/O callbacks** that have completed their operations. These callbacks might be associated with tasks like **reading from or writing to files**, **network communication**, or **database queries**.
  3. **Idle, Prepare Phase**: Internal phase used for internal purposes (not usually important to developers).
  4. **Poll Phase**: This phase is responsible for checking for new I/O events (e.g., incoming network requests, file system events) and preparing them for processing.
  6. **Check Phase**: Executes callbacks set by `setImmediate()`.
  7. **Close Callbacks Phase**: Executes callbacks for close events (e.g., `socket.on('close')`).
- **Microtask Queue Execution**: After each phase of the **Event Loop** (such as the Poll Phase or I/O Callbacks Phase), the **microtask queue** is checked, and any pending microtasks like **Promise** callbacks are executed before moving to the next phase.
  
  ---

- **Execution Flow**: Here's how the **Event Loop** and **Microtask Queue** work together:

    1. **Start a Cycle**: The **Event Loop** starts at the beginning of a cycle.

    2. **Execute Current Script**: The `JavaScript` code runs **synchronously until completion**, including function calls and synchronous operations.

    3. **Enter Event Loop Phases**: After the current script executes:

        * The Event Loop goes through each phase (Timers, I/O Callbacks, Poll, etc.) and executes any relevant callbacks in those phases.
    4. **Process Microtask Queue**: After completing a phase, the **Event Loop** checks the **Microtask Queue**:
        * If there are any microtasks like resolved Promises, it executes those callbacks.
        * The **Microtask Queue** is processed until it's empty before moving to the next Event Loop phase.
    5. **Repeat**: The **Event Loop** continues this process in a loop, handling events, timers, I/O, and microtasks as they arise.

## Using `setTimeout`

In [2]:
// Using 'setTimeout'

/*
'setTimeout' is used to execute a task after a given time passes.
*/

setTimeout(
    () => console.log("Hello World"),
    2000 // 2000 = 2 seconds
);

Timeout {
  _idleTimeout: 2000,
  _idlePrev: [TimersList],
  _idleNext: [TimersList],
  _idleStart: 2945,
  _onTimeout: [Function (anonymous)],
  _timerArgs: undefined,
  _repeat: null,
  _destroyed: false,
  [Symbol(refed)]: true,
  [Symbol(kHasPrimitive)]: false,
  [Symbol(asyncId)]: 19,
  [Symbol(triggerId)]: 16
}

Hello World


## Using `Promise`s

- In JavaScript, a `Promise` is an object that represents the eventual completion (or failure) of an **asynchronous** operation and its resulting value. It's a way to handle asynchronous tasks like data fetching or file reading more cleanly than using callbacks.

```javascript
let myPromise = new Promise((resolve, reject) => {
    let success = true; // Simulate success or failure
    if (success) {
        resolve("The operation was successful!");
    } else {
        reject("Something went wrong.");
    }
});

myPromise.then(result => {
    console.log(result); // Logs: The operation was successful!
}).catch(error => {
    console.log(error); // Logs: Something went wrong.
});


In [3]:
// A simple function that returns a Promise

function printWithDelay(d) {
    return new Promise((resolve, reject) => {
        if (d < 0) {
            reject('Delay must be a non-negative number'); // Reject the promise with an error message if the delay is negative
        } else {
            setTimeout(
                () => resolve('Time out'),
                d
            ); // Resolve the promise after the specified delay
        }
    });
}

// Call the function and handle the promise
printWithDelay(1000)
    .then(res => console.log(res)) // Handle the resolved promise
    .catch(err => console.error(err)); // Handle the rejected promise
printWithDelay(-1)
    .then(res => console.log(res)) // Handle the resolved promise
    .catch(err => console.error(err)); // Handle the rejected promise

Promise { <pending> }

Delay must be a non-negative number


Time out


## Using **Promise-Callback** Combination

- Instead of using `then-catch` externally, here it is used inside the function and we only send an arrow function (lambda) to handle success or failure situations; like when we were using `readFile`, `writeFile`, ....

In [4]:
// A promise is used inside a fucntion and gets callbacks for success and error

function printWithDelayAsync(d, cb) {
    // Defining the promise
    const promise = new Promise((resolve, reject) => {
        if (d < 0) {
            reject('Delay must be a non-negative number'); // Reject the promise with an error message if the delay is negative
        } else {
            setTimeout(
                () => resolve('Time out'),
                d
            ); // Resolve the promise after the specified delay
        }
    });

    // Using then-catch style on promise
    promise.then((res) => {
        cb(null, res); // Pass the result to the callback as the second argument
    })
    .catch((err) => {
        cb(err, null); // Pass the error to the callback as the first argument
    });
}

// Call the function and send it the callback
printWithDelayAsync(1000, (err, res) => {
    if (err) {
        console.error(err); // Log the error if it exists
        return;
    }
    console.log("Callback result", res); // Log the result if there's no error
});

printWithDelayAsync(-1, (err, res) => {
    if (err) {
        console.error(err); // Log the error if it exists
        return;
    }
    console.log("Callback result", res); // Log the result if there's no error
});


Delay must be a non-negative number


Callback result Time out


## Using Async-await

- An `async` functions return its result in a **promise**. If an exception happens, it can be caught in `catch`, otherwise `then` will get the result.

In [5]:
// Using promises with Async-await functions

// To define an asynchronous function use 'async' keyword before 'function'. 
// The result will be returned in a 'Promise'
async function awaitForPrint(d) {
    // Await for the returned promise.
    const message = await printWithDelay(d); // Await for the result of printWithDelay
    console.log(message); // Log the message
    return true; // Return true if no errors occur
}

// Call the async function and handle the promise
awaitForPrint(1000)
    .then(res => console.log('async run finished:', res))
    .catch(err => console.error('Caught an error:', err));

// Call the async function with a negative delay to trigger an error
awaitForPrint(-1)
    .then(res => console.log('async run finished:', res))
    .catch(err => console.error('Caught an error:', err));

Promise { <pending> }

Caught an error: Delay must be a non-negative number


Time out
async run finished: true


## Using Events

In [6]:
// Creating events

const eventEmitter = new EventEmitter();

// Add a permanent listener for the 'event1' event
eventEmitter.on('event1', (arg) => {
    console.log('event1 occurred:', arg);
});

// Add a one-time listener for the 'event2' event
eventEmitter.once('event2', () => {
    console.log('event2 occurred.');
});

EventEmitter {
  _events: [Object: null prototype] {
    event1: [Function (anonymous)],
    event2: [Function: bound onceWrapper] { listener: [Function (anonymous)] }
  },
  _eventsCount: 2,
  _maxListeners: undefined,
  [Symbol(kCapture)]: false
}

In [7]:
// Emitting

eventEmitter.emit('event1', 1);
eventEmitter.emit('event2');

event1 occurred: 1
event2 occurred.


true